# CFD Geometry — Google Colab

Generate aligned **building** and **tree** STL meshes for urban wind / OpenFOAM studies using OpenStreetMap data.

| Step | Task |
|------|------|
| 1 | Install package and map tools |
| 2 | Draw your study area |
| 3 | Download OSM data and build STLs |
| 4 | Inspect files and 3D preview |

---

### Open this notebook correctly

Use **[Open in Colab](https://colab.research.google.com/github/Omokayode/CFDGeometry/blob/main/notebooks/colab_quickstart.ipynb)** from GitHub. **Do not** use an old copy saved in Google Drive.

**Tips:** Keep the drawn rectangle small (city-block scale). After opening: **Runtime → Restart session**, then run cells in order.

**Notebook version:** `cfd-colab-v6`


## 1. Install

Installs CFDGeometry from GitHub without upgrading Colab's numpy (required for OSM download).


In [ ]:
import subprocess
import sys

_GIT = "git+https://github.com/Omokayode/CFDGeometry.git@main"
_WIDGETS = (
    "ipywidgets>=7.6,<9",
    "ipyleaflet>=0.17",
    "jupyterlab_widgets>=1.0.5,<4",
    "plotly>=5.18",
)
_STRATEGY = ("--upgrade-strategy", "only-if-needed")


def pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip(*_WIDGETS, *_STRATEGY)
pip("--upgrade", "--no-cache-dir", "--no-deps", f"{_GIT}#egg=cfd-geometry")
pip("osmnx>=1.9", "requests>=2.28", *_STRATEGY)
pip(
    "geopandas>=0.14",
    "rasterio>=1.3",
    "trimesh>=4.0",
    "mapbox-earcut>=1.0",
    "scipy>=1.11",
    "shapely>=2.0",
    "pyproj>=3.6",
    "pandas>=2.0",
    *_STRATEGY,
)

from google.colab import output

output.enable_custom_widget_manager()

import cfd_geometry
from cfd_geometry.domain import DomainConfig, build_domain  # noqa: F401

print("cfd_geometry", cfd_geometry.__version__)
print("Colab widgets: enabled")
print("notebook", "cfd-colab-v6")


## 2. Define study area

Pan/zoom the map, draw a **rectangle**, then click **Use this extent**.

Optional: set `PLACE` (geocoded name) or `CENTER = (latitude, longitude)`.


In [ ]:
from cfd_geometry.notebook import select_extent

PLACE = "Milwaukee, Wisconsin, USA"
CENTER = None  # e.g. (43.0389, -87.9065)

selector = select_extent(place=PLACE if CENTER is None else None, center=CENTER)
selector


## 3. Confirm extent

WGS84 bounding box (west, south, east, north) used for download and CLI.


In [ ]:
if selector.bbox is None:
    raise RuntimeError("Draw a rectangle on the map, then click 'Use this extent'.")

bbox = selector.bbox
print(f"west={bbox.west:.6f}  south={bbox.south:.6f}")
print(f"east={bbox.east:.6f}  north={bbox.north:.6f}")
print(
    f"\nCLI equivalent:\n"
    f"cfd-geometry domain -o data --bbox {bbox.west:.6f} {bbox.south:.6f} "
    f"{bbox.east:.6f} {bbox.north:.6f}"
)


## 4. Build geometry

Downloads OSM **buildings** and **trees** for the box, then writes aligned STL files under `data/output/`.


In [ ]:
from pathlib import Path

from cfd_geometry.domain import DomainConfig, build_domain

result = build_domain(
    DomainConfig(
        output_dir=Path("data"),
        bbox=selector.bbox,
        run_download=True,
        download_layers=("buildings", "trees"),
        build_buildings=True,
        build_trees=True,
        build_highways=False,
        build_terrain=False,
        height_source="composite",
    )
)
result.stl_files


## 5. Output files


In [ ]:
from pathlib import Path

out = Path("data/output")
for path in sorted(out.glob("*.stl")):
    size_kib = path.stat().st_size / 1024
    print(f"{path.name:24s} {size_kib:8.1f} KiB")


## 6. 3D preview

Interactive Plotly view (rotate/zoom). Reduce `max_triangles` if the plot is slow.


In [ ]:
from cfd_geometry.notebook.visualize import plot_domain_stls

plot_domain_stls(result, layers=("buildings", "trees"), max_triangles=8000)
